# Comparable Companies Valuation 

**Question:** what is a good enterprise value range for a company with approximately $5bn EBITDA in the tech sector.

**Method:** pull real data for 8 large-cap tech peers, compute their EV/EBITDA multiples, and apply the sector's median to the target's EBITDA.



## 1. Data collection

Download the main data from yfinance, with handling errors for missing tickers.

In [1]:
import yfinance as yf
import pandas as pd

tickers = ["MSFT", "ORCL", "ADBE", "CRM", "SAP", "NOW", "INTU", "WDAY"]

rows = []
failed = []

for t in tickers:
    try:
        info = yf.Ticker(t).info

        ev = info.get("enterpriseValue")
        ebitda = info.get("ebitda")
        sector = info.get("sector")
        
    except Exception:
        failed.append(t)
        continue

    if ev is None or ebitda is None:
        failed.append(t)
        continue
        
    rows.append({"ticker": t, "sector": sector, "ebitda": ebitda, "ev": ev})

data = pd.DataFrame(rows)
print ("Failed", failed)
data

Failed []


,ticker,sector,ebitda,ev
0,MSFT,Technology,184457003008,2947933536256
1,ORCL,Technology,30493999104,545083654144
2,ADBE,Technology,9729000448,88789696512
3,CRM,Technology,12894999552,166755090432
4,SAP,Technology,11603999744,187414167552
5,NOW,Technology,2888000000,106897670144
6,INTU,Technology,6409999872,75438415872
7,WDAY,Technology,1512000000,32895799296


## 2. Database (SQLite)

Store the data in SQLite as two tables - a reference table and a financials table - linked by the ticker to reproduce a real SQL schema and allow a join.

In [2]:
companies = data[["ticker", "sector"]].copy()
financials = data[["ticker", "ev", "ebitda"]].copy()

## 3. Join & load

Join the 2 tables by a JOIN SQL and load the results in pandas.

In [3]:
import sqlite3

conn = sqlite3.connect("comps.db")
companies.to_sql("companies", conn, index=False, if_exists="replace")
financials.to_sql("financials", conn, index=False, if_exists="replace")

df = pd.read_sql("""
    SELECT companies.ticker, companies.sector, financials.ev, financials.ebitda
    FROM companies
    JOIN financials ON companies.ticker = financials.ticker
""", conn)
df

,ticker,sector,ev,ebitda
0,MSFT,Technology,2947933536256,184457003008
1,ORCL,Technology,545083654144,30493999104
2,ADBE,Technology,88789696512,9729000448
3,CRM,Technology,166755090432,12894999552
4,SAP,Technology,187414167552,11603999744
5,NOW,Technology,106897670144,2888000000
6,INTU,Technology,75438415872,6409999872
7,WDAY,Technology,32895799296,1512000000


## 4. Valuation

Apply the sector's median EV/EBITDA to the target for a central estimate, and the interquartile range for a valuation range robust to outliers.

In [4]:
target_ebitda = 5_000_000_000
target_sector = "Technology"

df["ev_ebitda"] = df["ev"] / df["ebitda"]
comps_sector = df[df["sector"] == target_sector]
multiple_median = comps_sector["ev_ebitda"].median()
ev_estimated = multiple_median * target_ebitda 

print(f"EV estimated : {ev_estimated/1e9:.2f} billions of USD")

multiple_low = comps_sector["ev_ebitda"].quantile(0.25)
multiple_high = comps_sector["ev_ebitda"].quantile(0.75)

ev_low = multiple_low * target_ebitda
ev_high = multiple_high * target_ebitda

print(f"Valuation range : {ev_low/1e9:.2f} - {ev_high/1e9:.2f} billions of USD")

EV estimated : 80.33 billions of USD
Valuation range : 63.21 - 94.23 billions of USD


## 5. Result & limitation 

**Result:** a tech company with approximately USD 5bn EBITDA would be valued between USD 63bn and USD 94bn.

**Limits:** 
- Limited sample is too small to give a reliable median because with only 8 companies, one extreme value would move the median.
- Using only one multiple while a serious analysis would cross several multiples.
- All 8 companies sit in the same "Technology" sector, but this mixes very different business models (software, IT, services...) that don't trade on the same multiples.
